# Notebook 8: The Delta Optimizer (Resource Adjustment Engine)

## 1. Introduction
This notebook implements the core prescriptive logic of the project. Unlike a general search, this is a **Delta Optimizer**. 

Instead of finding *any* configuration that hits the target, this engine analyzes the current baseline and suggests the **minimum necessary adjustment** to align production with the plan.

### The Adjustment Strategy:
1. **Analyze State**: Check if the baseline is Overproducing, Underproducing, or Balanced.
2. **Directional Search**:
    - **Underproduction** $\rightarrow$ Search only for configurations with $\text{resources} \ge \text{baseline}$.
    - **Overproduction** $\rightarrow$ Search only for configurations with $\text{resources} \le \text{baseline}$.
    - **Balanced** $\rightarrow$ Suggest no change.
3. **Proximity Scoring**: Rank configurations by the **smallest total change** from the baseline that successfully hits the target.

In [5]:
import pandas as pd
import numpy as np
import joblib
import itertools

# Load the saved model pipeline
MODEL_PATH = '../models/production_model.joblib'
model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


## 2. Integrating Baseline & Constraints

We bring together the feature building logic (Notebook 6) and the operational bounds (Notebook 7).

In [6]:
# --- Logic from Notebook 6: Baseline Feature Builder ---
def build_prescriptive_features(raw_inputs):
    features = raw_inputs.copy()
    workers = max(raw_inputs.get('no_of_workers', 1), 1)
    features['machine_worker_ratio'] = raw_inputs.get('machine_count', 0) / workers
    features['avg_hours_per_worker'] = raw_inputs.get('working_hours', 0) / workers
    defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
    for col, value in defaults.items():
        if col not in features: features[col] = value
    return pd.DataFrame([features])

# --- Logic from Notebook 7: Constraints ---
RESOURCE_CONSTRAINTS = {
    'no_of_workers': {'min': 8, 'max': 32, 'step': 1},
    'working_hours': {'min': 200.0, 'max': 1600.0, 'step': 8.0},
    'machine_count': {'min': 2, 'max': 10, 'step': 1}
}

def is_feasible(config):
    for resource, bounds in RESOURCE_CONSTRAINTS.items():
        val = config.get(resource)
        if val is None or val < bounds['min'] or val > bounds['max']:
            return False
    return True

print("Baseline and Constraint logic integrated.")

Baseline and Constraint logic integrated.


## 3. The Delta Optimizer Implementation

The `DeltaOptimizer` class implements the specific business logic: if we are overproducing, only look for ways to reduce; if underproducing, only look for ways to increase.

In [7]:
class DeltaOptimizer:
    def __init__(self, model, constraints):
        self.model = model
        self.constraints = constraints
        self.w_gap = 1.0  # Priority on hitting the target
        self.w_delta = 0.1 # Priority on smallest change from baseline

    def generate_search_space(self):
        workers = range(self.constraints['no_of_workers']['min'],
                        self.constraints['no_of_workers']['max'] + 1,
                        self.constraints['no_of_workers']['step'])
        hours = np.arange(self.constraints['working_hours']['min'],
                          self.constraints['working_hours']['max'] + 0.1,
                          self.constraints['working_hours']['step'])
        machines = range(self.constraints['machine_count']['min'],
                         self.constraints['machine_count']['max'] + 1,
                         self.constraints['machine_count']['step'])
        return list(itertools.product(workers, hours, machines))

    def calculate_resource_cost(self, df):
        """Vectorized cost calculation for the entire dataframe."""
        norm_workers = df['no_of_workers'] / 32
        norm_hours = df['working_hours'] / 1500
        norm_machines = df['machine_count'] / 7
        return (norm_workers + norm_hours + norm_machines) / 3

    def optimize(self, current_resources, planned_qty, context_vars):
        # 1. Get Baseline Prediction
        full_baseline_inputs = {**current_resources, **context_vars}
        baseline_feat = build_prescriptive_features(full_baseline_inputs)
        baseline_pred = self.model.predict(baseline_feat)[0]

        # 2. Determine State and Search Direction
        gap = baseline_pred - planned_qty
        if abs(gap) <= 2:
            return "Balanced", None

        state = "Overproduction" if gap > 2 else "Underproduction"

        # 3. Generate all combinations at once (FAST)
        search_space = self.generate_search_space()
        df_scenarios = pd.DataFrame(search_space, columns=['no_of_workers', 'working_hours', 'machine_count'])

        # 4. Directional Filter (FAST)
        if state == "Overproduction":
            df_scenarios = df_scenarios[
                (df_scenarios['no_of_workers'] <= current_resources['no_of_workers']) &
                (df_scenarios['working_hours'] <= current_resources['working_hours']) &
                (df_scenarios['machine_count'] <= current_resources['machine_count'])
            ].copy()
        elif state == "Underproduction":
            df_scenarios = df_scenarios[
                (df_scenarios['no_of_workers'] >= current_resources['no_of_workers']) &
                (df_scenarios['working_hours'] >= current_resources['working_hours']) &
                (df_scenarios['machine_count'] >= current_resources['machine_count'])
            ].copy()

        # 5. Add context variables to all rows (FAST)
        for col, val in context_vars.items():
            df_scenarios[col] = val

        # 6. Calculate derived features for all rows at once (FAST)
        df_scenarios['machine_worker_ratio'] = df_scenarios['machine_count'] / df_scenarios['no_of_workers']
        df_scenarios['avg_hours_per_worker'] = df_scenarios['working_hours'] / df_scenarios['no_of_workers']

        # 7. Fill baseline defaults
        defaults = {'week_no': 26, 'week_of_month': 3, 'financial_month': 'Sep', 'bottleneck_flag': False, 'rate': 5266.92}
        for col, value in defaults.items():
            if col not in df_scenarios.columns:
                df_scenarios[col] = value

        # 8. ONE SINGLE PREDICTION for all rows (LIGHTNING FAST)
        predictions = self.model.predict(df_scenarios)

        # 9. Vectorized Scoring
        df_scenarios['predicted_qty'] = predictions
        df_scenarios['gap'] = abs(planned_qty - df_scenarios['predicted_qty'])
        df_scenarios['cost'] = self.calculate_resource_cost(df_scenarios)
        df_scenarios['score'] = (self.w_gap * df_scenarios['gap']) + (self.w_delta * df_scenarios['cost'])

        return state, df_scenarios.sort_values('score').head(5)

## 4. Testing the Delta Logic

We will test the system by providing a current resource set and a target quantity.

In [9]:
context = {
    'item_name': 'DIYANA - L - BROWN',
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Gents',
    'size_id': 10,
    'grade': 'A',
    'order_category': 'Others'
}

current_resources = {
    'no_of_workers': 15,
    'working_hours': 623.96,
    'machine_count': 4
}

# Create the optimizer using the loaded model and constraints
optimizer = DeltaOptimizer(model, RESOURCE_CONSTRAINTS)

# Try a target that causes underproduction (e.g., target 60)
target = 60
state, recs = optimizer.optimize(current_resources, target, context)

print(f"State: {state}")
if recs is not None:
    print(f"Top recommendations to reach {target} units:")
    display(recs)
else:
    print("No change needed!")

State: Underproduction
Top recommendations to reach 60 units:


,no_of_workers,working_hours,machine_count,item_name,factory_unit,product_family,size_id,grade,order_category,machine_worker_ratio,avg_hours_per_worker,week_no,week_of_month,financial_month,bottleneck_flag,rate,predicted_qty,gap,cost,score
12044,15,1048.0,4,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.266667,69.866667,26,3,Sep,False,5266.92,59.998894,0.001106,0.579615,0.059067
13664,16,1080.0,4,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.250000,67.500000,26,3,Sep,False,5266.92,60.001793,0.001793,0.597143,0.061507
24977,23,1280.0,4,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.173913,55.652174,26,3,Sep,False,5266.92,59.986017,0.013983,0.714504,0.085433
13343,16,792.0,7,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.437500,49.500000,26,3,Sep,False,5266.92,60.018414,0.018414,0.676000,0.086014
11698,15,736.0,9,DIYANA - L - BROWN,Factory Unit 01,Gents,10,A,Others,0.600000,49.066667,26,3,Sep,False,5266.92,60.011848,0.011848,0.748377,0.086686


## 5. Analysis of Results
Looking at the results for a target of 60 units:

- Baseline State: The system correctly identified an Underproduction state.
- Optimal Adjustment: The Delta Optimizer found that by increasing working hours to 1048.0 (while keeping workers at 15 and machines at 4), the predicted output reaches 59.99 units.
- Efficiency: This represents a near-perfect match to the target with the smallest possible increase in resource cost.

This proves the Delta Optimizer can surgically adjust resources to hit a specific production goal.